# Test `similarity_score()` from `cosine_similarity.py`

This notebook imports `similarity_score()` from your local `cosine_similarity.py` file
and runs a few quick tests.

**Setup:** Put this notebook in the **same folder** as `cosine_similarity.py`.


In [ ]:
# pip install sentence-transformers

## Import from `cosine_similarity.py`

In [14]:
import sys, os, importlib
sys.path.append("../")  # keep if you need it

import cosine_similarity
print("Loaded from:", cosine_similarity.__file__)

importlib.reload(cosine_similarity)

from cosine_similarity import similarity_score, similarity_many


Loaded from: c:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\unified_schema\testings\..\cosine_similarity.py


In [15]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

def get_best_device() -> str:
    # Preference: CUDA (NVIDIA) -> MPS (Apple Silicon) -> CPU
    try:
        import torch
    except Exception:
        return "cpu"

    if torch.cuda.is_available():
        return "cuda"

    # Apple Silicon (only if torch was built with MPS)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"

    return "cpu"
device = get_best_device()
device


2.9.1+cu128
CUDA available: True
CUDA runtime: 12.8
GPU: NVIDIA GeForce RTX 3080 Laptop GPU


'cuda'

## Quick sanity checks

In [17]:
pairs = [
    ("A cat sits on the mat.", "A feline is sitting on a rug."),
    ("I love pizza.", "The stock market fell today."),
    ("Toronto is in Canada.", "Ottawa is the capital of Canada."),
    ("", "Non-empty"),
]

for a, b in pairs:
    print(f"{similarity_score(a, b, device=device):.4f} | {a!r}  <->  {b!r}")

0.6278 | 'A cat sits on the mat.'  <->  'A feline is sitting on a rug.'
-0.0075 | 'I love pizza.'  <->  'The stock market fell today.'
0.6309 | 'Toronto is in Canada.'  <->  'Ottawa is the capital of Canada.'
0.0000 | ''  <->  'Non-empty'


In [19]:
query = "cats sitting on rugs"
candidates = [
    "a cat is sitting on a mat",
    "financial markets closed lower today",
    "feline on a carpet",
    "",  # empty -> score 0.0
]

device = "cuda"  # or get_best_device()

scores = similarity_many(query, candidates, device=device)

# print all scores
for s, c in zip(scores, candidates):
    print(f"{s:.4f} | {c!r}")

# top-k results
k = 2
top_idx = scores.argsort()[::-1][:k]
print("\nTOP", k)
for i in top_idx:
    print(f"{scores[i]:.4f} | {candidates[i]!r}")

# sanity check: first candidate should match single-pair result
print("\ncheck:",
      similarity_score(query, candidates[0], device=device),
      scores[0])


0.6681 | 'a cat is sitting on a mat'
0.0140 | 'financial markets closed lower today'
0.6567 | 'feline on a carpet'
0.0000 | ''

TOP 2
0.6681 | 'a cat is sitting on a mat'
0.6567 | 'feline on a carpet'

check: 0.6681479811668396 0.6681481
